# Chapter 04: Topological Foundations and Differential Geometry

**Source orientation:** PDF pages 37-50 of *Mathematical Foundations of Geometric Deep Learning*. This notebook is a standalone computational lesson. It uses the source span for terminology and scope, but all prose, examples, diagrams, and checks here are original.

**Chapter goal:** Build a computational working model for spaces that are locally Euclidean but globally structured. By the end, you should be able to inspect openness, continuity, topological equivalence, Euler characteristic, charts, tangent spaces, geodesics, exponential/logarithmic maps, gauges, and the manifold hypothesis as concrete arrays, maps, surfaces, and invariants.

## Translation guide: computational dictionary

| Source idea | Computational object in this notebook | What to inspect |
| --- | --- | --- |
| Open set | membership test plus local balls | an interior point has a positive radius ball inside the set; a boundary point does not |
| Topology | finite collection of subsets closed under unions and finite intersections | the axioms are checks on collections of subsets, not on distances |
| Continuity | preimage test for open target sets | continuous maps pull open output windows back to open input regions |
| Homeomorphism | bijective continuous map with continuous inverse | deformation without tearing, gluing, or identifying points |
| Homotopy | continuous deformation of maps or spaces | collapse maps can be continuous while losing inverse information |
| Euler characteristic | `V - E + F` on a triangulated surface | shape changes that preserve topology keep the same count |
| Chart and atlas | local coordinate map and transition map | charts overlap through smooth coordinate changes |
| Tangent space | local linear plane of directions | geodesic directions and derivatives live in this linear space |
| Exponential/log maps | conversions between tangent vectors and nearby manifold points | `exp_p(log_p(q))` returns `q` away from cut-locus pathologies |
| Gauge | a selected basis for each tangent space | changing the basis must not change intrinsic tangent vectors or norms |
| Manifold hypothesis | low-dimensional latent coordinates embedded in high-dimensional observations | valid data occupies a thin structured subset of ambient space |

## Visual storyboard and library routing

1. `open-set-neighborhood-test.png` uses Matplotlib and a finite-topology checker to separate metric intuition from topology axioms.
2. `continuity-preimage-open-sets.png` uses Matplotlib to compare a continuous square map with a discontinuous step map using preimages.
3. `homeomorphism-homotopy-tracker.png` uses Matplotlib plus Ripser/Persim persistence summaries to show a homeomorphism-like deformation, a collapse map, and the loop information lost by collapse.
4. `euler-characteristic-surface-invariants.png` and `euler-characteristic-table.csv` use Trimesh to compute `V`, `E`, `F`, and `chi` for sphere-like, cube-like, and torus-like meshes.
5. `chart-transition-tangent-space.png` uses SymPy and Matplotlib to expose stereographic chart transitions and the tangent-plane linearization.
6. `sphere-geodesics-exp-log-charts.html` uses Plotly for a rotatable sphere, tangent plane, geodesic, and exp/log construction.
7. `gauge-frame-equivariance.png` uses NumPy linear algebra and Matplotlib to show two gauges for the same tangent vector and the coordinate rotation relating them.
8. `manifold-hypothesis-latent-lab.png` and `manifold-hypothesis-local-dimension.csv` use a synthetic image generator plus local PCA to contrast a low-dimensional data manifold with random ambient pixels.
9. `topology-manifold-invariants.json` records the checks that make the visuals more than illustrations.


In [ ]:
from pathlib import Path
import sys

BOOK_ROOT = Path.cwd()
for candidate in [BOOK_ROOT, *BOOK_ROOT.parents]:
    if (candidate / "00-book-index.ipynb").exists() and (candidate / "utils").exists():
        BOOK_ROOT = candidate
        break
else:
    raise RuntimeError("Could not find the MFGDL book root")

if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / "chapter-04"
for folder in ["figures", "html", "checks", "tables"]:
    target = ARTIFACT_ROOT / folder
    target.mkdir(parents=True, exist_ok=True)
    for stale in target.iterdir():
        if stale.is_file() and stale.suffix.lower() in {".png", ".svg", ".html", ".json", ".csv"}:
            stale.unlink()

ARTIFACT_ROOT.relative_to(BOOK_ROOT)


In [ ]:
import json
import math
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.collections import LineCollection
from matplotlib.patches import Circle, Rectangle
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
import plotly.graph_objects as go
import sympy as sp
import trimesh
from scipy.spatial import cKDTree
from scipy.linalg import svd
from ripser import ripser

try:
    from persim import plot_diagrams
except Exception:
    plot_diagrams = None

from utils.artifacts import display_artifact, save_json, save_matplotlib, save_plotly_html, save_table_csv
from utils.notebook_checks import assert_chapter_artifacts, assert_nonblank_image

rng = np.random.default_rng(7)
plt.rcParams.update({
    "figure.dpi": 130,
    "savefig.facecolor": "white",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 10,
})

artifact_paths = []
checks = {}


def remember(path):
    path = Path(path)
    artifact_paths.append(path)
    return path


def powerset(items):
    items = tuple(items)
    return [frozenset(s) for r in range(len(items) + 1) for s in combinations(items, r)]


def is_topology(X, collection):
    X = frozenset(X)
    T = {frozenset(s) for s in collection}
    finite_subsets = powerset(T)
    contains_bounds = frozenset() in T and X in T
    union_closed = all(frozenset().union(*family) in T for family in finite_subsets)
    intersection_closed = True
    for family in finite_subsets:
        if not family:
            continue
        inter = set(X)
        for subset in family:
            inter &= set(subset)
        if frozenset(inter) not in T:
            intersection_closed = False
            break
    return contains_bounds and union_closed and intersection_closed


COLOR_A = "#2c7fb8"
COLOR_B = "#fdae61"
COLOR_C = "#31a354"
COLOR_D = "#756bb1"
COLOR_BAD = "#d95f0e"


## 1. Open Sets: Distance Is One Route, Not the Definition

The chapter starts by separating a bare set from a topological space. A set only answers membership questions. A topology adds a chosen family of subsets called open sets. In a metric space, open balls generate the familiar Euclidean topology, but the axioms themselves only talk about collections of subsets: include the empty set and whole space, close under arbitrary unions, and close under finite intersections.

The left panel below uses the metric intuition: an interior point has some positive-radius ball contained in the set, while a boundary point does not. The right panel uses a finite set to make the axioms inspectable without drawing distances at all.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
ax = axes[0]
ax.set_title("Metric open-set test")
set_radius = 1.55
interior = np.array([0.35, 0.2])
boundary = np.array([set_radius, 0.0])
interior_margin = set_radius - np.linalg.norm(interior)

ax.add_patch(Circle((0, 0), set_radius, facecolor="#d8ecf6", edgecolor=COLOR_A, lw=2, alpha=0.9))
ax.add_patch(Circle(interior, interior_margin * 0.62, facecolor="none", edgecolor=COLOR_C, lw=2, ls="--"))
ax.add_patch(Circle(boundary, 0.28, facecolor="none", edgecolor=COLOR_BAD, lw=2, ls=":"))
ax.scatter(*interior, s=60, color=COLOR_C, zorder=5)
ax.scatter(*boundary, s=60, color=COLOR_BAD, zorder=5)
ax.annotate("interior point\npositive ball fits", interior + np.array([0.15, 0.25]), color=COLOR_C)
ax.annotate("boundary point\nno positive ball fits", boundary + np.array([-1.15, -0.45]), color=COLOR_BAD)
ax.set_aspect("equal")
ax.set_xlim(-1.9, 2.1)
ax.set_ylim(-1.8, 1.8)
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.grid(alpha=0.2)

ax = axes[1]
ax.set_title("Finite topology: open sets are selected subsets")
X = frozenset({"a", "b", "c"})
trivial = {frozenset(), X}
nested = {frozenset(), frozenset({"a"}), frozenset({"a", "b"}), X}
discrete = set(powerset(X))
collections = [("trivial", trivial, 2.3), ("nested", nested, 1.1), ("discrete", discrete, -0.1)]
for label, collection, y in collections:
    ax.text(-0.25, y, label, ha="right", va="center", weight="bold")
    for j, subset in enumerate(sorted(collection, key=lambda s: (len(s), sorted(s)))):
        x0 = j * 0.9
        ax.add_patch(Rectangle((x0 - 0.3, y - 0.22), 0.6, 0.44, facecolor="#f7f7f7", edgecolor="#444444", lw=1))
        label_text = "{}" if len(subset) == 0 else "{" + ",".join(sorted(subset)) + "}"
        ax.text(x0, y, label_text, ha="center", va="center", fontsize=8)
    ax.text(7.0, y, "axioms pass" if is_topology(X, collection) else "fails", color=COLOR_C)
ax.set_xlim(-1.35, 8.2)
ax.set_ylim(-0.7, 2.9)
ax.axis("off")
fig.suptitle("Open sets: local metric balls and abstract topology axioms", y=1.02)
fig.tight_layout()

path = remember(save_matplotlib(fig, "chapter-04", "open-set-neighborhood-test.png"))
plt.close(fig)

checks["open_sets"] = {
    "interior_margin_positive": float(interior_margin),
    "boundary_margin_zero": float(set_radius - np.linalg.norm(boundary)),
    "finite_topology_axioms": {
        "trivial": is_topology(X, trivial),
        "nested": is_topology(X, nested),
        "discrete": is_topology(X, discrete),
    },
}
assert checks["open_sets"]["interior_margin_positive"] > 0
assert abs(checks["open_sets"]["boundary_margin_zero"]) < 1e-12
assert all(checks["open_sets"]["finite_topology_axioms"].values())
display_artifact(path, width=900)


## 2. Continuity: Pull Open Sets Backward

For topological spaces, continuity is not defined by a graph being easy to draw. It is defined by a preimage rule: every open target set must pull back to an open source set. The square map passes this test on the window shown here. The step map fails because the preimage of an open output interval contains its jump point as a one-sided endpoint, so no small open ball around that point stays inside the preimage.


In [ ]:
x = np.linspace(-2.2, 2.2, 800)
f = x**2
step = (x >= 0).astype(float)

fig, axes = plt.subplots(2, 2, figsize=(12, 6), sharex="col")
for ax in axes.ravel():
    ax.grid(alpha=0.2)

axes[0, 0].plot(x, f, color=COLOR_A, lw=2)
axes[0, 0].axhspan(0.25, 2.25, color=COLOR_B, alpha=0.25, label="open target interval")
axes[0, 0].set_title("continuous map $f(x)=x^2$")
axes[0, 0].set_ylabel("output")
axes[0, 0].legend(loc="upper center", frameon=False)

axes[1, 0].axhline(0, color="#888888", lw=1)
for lo, hi in [(-1.5, -0.5), (0.5, 1.5)]:
    axes[1, 0].plot([lo, hi], [0, 0], color=COLOR_C, lw=8, solid_capstyle="butt")
    axes[1, 0].scatter([lo, hi], [0, 0], s=60, facecolor="white", edgecolor=COLOR_C, zorder=4)
axes[1, 0].set_ylim(-0.35, 0.35)
axes[1, 0].set_yticks([])
axes[1, 0].set_title("preimage: two open intervals")
axes[1, 0].set_xlabel("input x")

axes[0, 1].plot(x, step, color=COLOR_BAD, lw=2)
axes[0, 1].axhspan(0.5, 1.5, color=COLOR_B, alpha=0.25)
axes[0, 1].set_title("discontinuous step map")

axes[1, 1].axhline(0, color="#888888", lw=1)
axes[1, 1].plot([0, 2.2], [0, 0], color=COLOR_BAD, lw=8, solid_capstyle="butt")
axes[1, 1].scatter([0], [0], s=70, facecolor=COLOR_BAD, edgecolor=COLOR_BAD, zorder=4, label="included jump point")
axes[1, 1].scatter([2.2], [0], s=60, facecolor="white", edgecolor=COLOR_BAD, zorder=4)
axes[1, 1].set_ylim(-0.35, 0.35)
axes[1, 1].set_yticks([])
axes[1, 1].set_title("preimage: $[0, \\infty)$ is not open in $R$")
axes[1, 1].set_xlabel("input x")
axes[1, 1].legend(loc="lower right", frameon=False)
fig.suptitle("Continuity is a preimage test for open sets", y=1.02)
fig.tight_layout()

path = remember(save_matplotlib(fig, "chapter-04", "continuity-preimage-open-sets.png"))
plt.close(fig)

checks["continuity_preimages"] = {
    "square_preimage_components": [[-1.5, -0.5], [0.5, 1.5]],
    "square_endpoints_map_to_target_boundary": [float((-1.5)**2), float((-0.5)**2), float(0.5**2), float(1.5**2)],
    "step_preimage_contains_jump_endpoint": True,
    "epsilon_ball_around_jump_inside_preimage": False,
}
assert np.allclose(checks["continuity_preimages"]["square_endpoints_map_to_target_boundary"], [2.25, 0.25, 0.25, 2.25])
assert checks["continuity_preimages"]["step_preimage_contains_jump_endpoint"]
assert not checks["continuity_preimages"]["epsilon_ball_around_jump_inside_preimage"]
display_artifact(path, width=900)


## 3. Homeomorphism, Homotopy, and What Collapse Loses

A homeomorphism is a reversible topological equivalence. It can stretch or bend, but it cannot identify two different points. A homotopy is a continuous deformation. A collapse map from a circle to a point is continuous as a map, but it is not a homeomorphism because it destroys injectivity and has no continuous inverse. The persistence panel is included as a computational warning: the sampled circle has a durable one-dimensional loop, while a filled disk does not.


In [ ]:
theta = np.linspace(0, 2 * np.pi, 220, endpoint=False)
circle = np.column_stack([np.cos(theta), np.sin(theta)])
ellipse = np.column_stack([1.45 * np.cos(theta), 0.72 * np.sin(theta)])
collapse_radii = [1.0, 0.72, 0.42, 0.14, 0.0]

r = np.sqrt(rng.uniform(0.0, 1.0, 260))
phi = rng.uniform(0.0, 2 * np.pi, 260)
disk = np.column_stack([r * np.cos(phi), r * np.sin(phi)])
circle_dgms = ripser(circle[::3], maxdim=1)["dgms"]
disk_dgms = ripser(disk, maxdim=1)["dgms"]


def max_lifetime(diagram):
    finite = diagram[np.isfinite(diagram[:, 1])]
    if finite.size == 0:
        return 0.0
    return float(np.max(finite[:, 1] - finite[:, 0]))


fig = plt.figure(figsize=(13, 4.6))
grid = fig.add_gridspec(1, 3, width_ratios=[1.05, 1.1, 1.0])
ax0 = fig.add_subplot(grid[0, 0])
ax1 = fig.add_subplot(grid[0, 1])
ax2 = fig.add_subplot(grid[0, 2])

ax0.plot(circle[:, 0], circle[:, 1], color=COLOR_A, lw=2, label="circle")
ax0.plot(ellipse[:, 0], ellipse[:, 1], color=COLOR_C, lw=2, label="stretched circle")
for k in range(0, len(theta), 28):
    ax0.plot([circle[k, 0], ellipse[k, 0]], [circle[k, 1], ellipse[k, 1]], color="#999999", lw=0.8, alpha=0.6)
ax0.set_title("homeomorphism-like stretch\npoints are not identified")
ax0.set_aspect("equal")
ax0.legend(frameon=False)
ax0.axis("off")

for j, radius in enumerate(collapse_radii):
    y_offset = 1.2 - j * 0.6
    pts = radius * circle * 0.25 + np.array([j * 0.75 - 1.5, y_offset])
    if radius > 0:
        ax1.plot(pts[:, 0], pts[:, 1], color=cm.viridis(j / 4), lw=2)
    else:
        ax1.scatter([j * 0.75 - 1.5], [y_offset], s=70, color=COLOR_BAD)
    ax1.text(j * 0.75 - 1.5, y_offset - 0.32, f"t={j/4:.2f}", ha="center", fontsize=8)
ax1.annotate("continuous collapse", xy=(0.5, 0.75), xytext=(-1.35, -1.0), arrowprops={"arrowstyle": "->"})
ax1.set_title("homotopy of maps can collapse\nbut the final map is not invertible")
ax1.set_aspect("equal")
ax1.axis("off")

if plot_diagrams is not None:
    plot_diagrams([circle_dgms[1], disk_dgms[1]], ax=ax2, labels=["circle H1", "disk H1"], show=False)
    ax2.set_title("persistent loop detector")
else:
    for label, dgms, color in [("circle", circle_dgms, COLOR_A), ("disk", disk_dgms, COLOR_B)]:
        dgm = dgms[1]
        ax2.scatter(dgm[:, 0], dgm[:, 1], label=label, color=color)
    ax2.legend(frameon=False)
    ax2.set_title("persistent loop detector")
ax2.grid(alpha=0.25)
fig.suptitle("Homeomorphism is reversible; collapse is continuous but loses topology", y=1.02)
fig.tight_layout()

path = remember(save_matplotlib(fig, "chapter-04", "homeomorphism-homotopy-tracker.png"))
plt.close(fig)

checks["homeomorphism_homotopy"] = {
    "ellipse_map_min_speed": float(np.min(np.linalg.norm(np.gradient(ellipse, axis=0), axis=1))),
    "collapse_many_angles_same_endpoint": True,
    "circle_max_h1_lifetime": max_lifetime(circle_dgms[1]),
    "disk_max_h1_lifetime": max_lifetime(disk_dgms[1]),
}
assert checks["homeomorphism_homotopy"]["collapse_many_angles_same_endpoint"]
assert checks["homeomorphism_homotopy"]["circle_max_h1_lifetime"] > checks["homeomorphism_homotopy"]["disk_max_h1_lifetime"]
display_artifact(path, width=950)


## 4. Euler Characteristic: A Global Count That Ignores Geometry

The Euler characteristic is a global topological invariant for the triangulated closed surfaces we use here. It does not care whether a sphere-like surface is round or cubical, but it does detect the handle of a torus. Trimesh gives us vertices and triangular faces; the code builds the unique edge set and verifies `chi = V - E + F`.


In [ ]:
def mesh_counts(mesh):
    faces = np.asarray(mesh.faces)
    edges = set()
    for tri in faces:
        for i, j in [(0, 1), (1, 2), (2, 0)]:
            a, b = sorted((int(tri[i]), int(tri[j])))
            edges.add((a, b))
    V = int(len(mesh.vertices))
    E = int(len(edges))
    F = int(len(faces))
    return {"vertices": V, "edges": E, "faces": F, "euler": V - E + F}


meshes = {
    "icosphere (sphere topology)": trimesh.creation.icosphere(subdivisions=2, radius=1.0),
    "triangulated cube (sphere topology)": trimesh.creation.box(extents=(1.8, 1.8, 1.8)),
    "torus (one handle)": trimesh.creation.torus(major_radius=1.0, minor_radius=0.32, major_sections=48, minor_sections=16),
}
records = []
for name, mesh in meshes.items():
    rec = {"surface": name, **mesh_counts(mesh), "trimesh_euler": int(mesh.euler_number), "watertight": bool(mesh.is_watertight)}
    records.append(rec)

table_path = remember(save_table_csv(records, "chapter-04", "euler-characteristic-table.csv"))

fig = plt.figure(figsize=(12, 4.2))
for idx, (name, mesh) in enumerate(meshes.items(), start=1):
    ax = fig.add_subplot(1, 3, idx, projection="3d")
    verts = np.asarray(mesh.vertices)
    faces = np.asarray(mesh.faces)
    collection = Poly3DCollection(verts[faces], alpha=0.82, linewidths=0.25, edgecolor="#333333")
    collection.set_facecolor([COLOR_A, COLOR_C, COLOR_B][idx - 1])
    ax.add_collection3d(collection)
    scale = float(np.max(np.abs(verts))) * 1.1
    ax.set_xlim(-scale, scale)
    ax.set_ylim(-scale, scale)
    ax.set_zlim(-scale, scale)
    ax.set_box_aspect((1, 1, 1))
    ax.view_init(elev=22, azim=-38)
    rec = records[idx - 1]
    ax.set_title(f"{name}\nV-E+F={rec['vertices']}-{rec['edges']}+{rec['faces']}={rec['euler']}", fontsize=9)
    ax.set_axis_off()
fig.suptitle("Euler characteristic from mesh combinatorics", y=1.02)
fig.tight_layout()

path = remember(save_matplotlib(fig, "chapter-04", "euler-characteristic-surface-invariants.png"))
plt.close(fig)

checks["euler_characteristic"] = records
for rec in records:
    assert rec["euler"] == rec["trimesh_euler"]
    assert rec["watertight"]
assert records[0]["euler"] == 2 and records[1]["euler"] == 2 and records[2]["euler"] == 0
pd.DataFrame(records)


In [ ]:
display_artifact(ARTIFACT_ROOT / "figures" / "euler-characteristic-surface-invariants.png", width=900)
display_artifact(table_path, width=900)


## 5. Charts, Tangent Spaces, Geodesics, and Exp/Log Maps

A manifold is locally Euclidean: near a point, a chart gives coordinates. Smooth manifolds require compatible chart transitions, so calculus does not depend on which valid chart was used. On the unit sphere, stereographic charts are a compact model for this idea. The transition between the north and south stereographic charts is an inversion map on the overlap, smooth away from the missing pole.

The tangent space is the local linear workspace. On the sphere, a tangent vector at `p` exponentiates by following the great-circle geodesic with that initial direction. The logarithmic map reverses that move for nearby non-antipodal points.


In [ ]:
u, v = sp.symbols("u v", real=True)
r2 = u**2 + v**2
transition = sp.Matrix([u / r2, v / r2])
jacobian = transition.jacobian([u, v])
det_jacobian = sp.simplify(jacobian.det())
checks["chart_transition"] = {
    "transition_map": [str(item) for item in transition],
    "jacobian_det": str(det_jacobian),
    "nonzero_on_overlap_except_origin": True,
}
assert sp.simplify(det_jacobian - (-1 / (u**2 + v**2)**2)) == 0

grid_values = np.linspace(-1.5, 1.5, 13)
segments_before = []
segments_after = []
for fixed in grid_values:
    pts = np.column_stack([np.linspace(-1.5, 1.5, 120), np.full(120, fixed)])
    pts = pts[np.linalg.norm(pts, axis=1) > 0.35]
    segments_before.append(pts)
    segments_after.append(pts / np.sum(pts**2, axis=1, keepdims=True))
    pts = np.column_stack([np.full(120, fixed), np.linspace(-1.5, 1.5, 120)])
    pts = pts[np.linalg.norm(pts, axis=1) > 0.35]
    segments_before.append(pts)
    segments_after.append(pts / np.sum(pts**2, axis=1, keepdims=True))

fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))
for ax, segments, title, color in [
    (axes[0], segments_before, "chart coordinates on overlap", COLOR_A),
    (axes[1], segments_after, "same points after transition", COLOR_C),
]:
    lc = LineCollection(segments, colors=color, linewidths=0.9, alpha=0.8)
    ax.add_collection(lc)
    ax.add_patch(Circle((0, 0), 0.35, facecolor="#f2f2f2", edgecolor="#888888", ls="--"))
    ax.set_xlim(-2.1, 2.1)
    ax.set_ylim(-2.1, 2.1)
    ax.set_aspect("equal")
    ax.grid(alpha=0.18)
    ax.set_title(title)
    ax.set_xlabel("u")
    ax.set_ylabel("v")

ax = axes[2]
xs = np.linspace(-0.95, 0.95, 160)
arc = np.sqrt(np.maximum(0, 1 - xs**2))
ax.plot(xs, arc, color=COLOR_A, lw=2, label="sphere cross-section")
ax.axhline(1, color=COLOR_C, lw=2, label="tangent line at north pole")
for eps in [0.2, 0.45, 0.7]:
    sphere_point = np.array([math.sin(eps), math.cos(eps)])
    tangent_point = np.array([eps, 1.0])
    ax.plot([0, sphere_point[0]], [1, sphere_point[1]], color=COLOR_B, lw=1.6)
    ax.scatter([sphere_point[0], tangent_point[0]], [sphere_point[1], tangent_point[1]], s=28)
    ax.plot([sphere_point[0], tangent_point[0]], [sphere_point[1], tangent_point[1]], color="#777777", ls=":", lw=0.9)
ax.set_title("tangent linearization\nerror grows with distance")
ax.set_aspect("equal")
ax.legend(frameon=False, fontsize=8)
ax.grid(alpha=0.18)
fig.suptitle("Charts and tangent spaces are local interfaces", y=1.02)
fig.tight_layout()

path = remember(save_matplotlib(fig, "chapter-04", "chart-transition-tangent-space.png"))
plt.close(fig)
display_artifact(path, width=950)


In [ ]:
def exp_sphere(p, tangent):
    p = np.asarray(p, dtype=float)
    tangent = np.asarray(tangent, dtype=float)
    norm = np.linalg.norm(tangent)
    if norm < 1e-12:
        return p.copy()
    direction = tangent / norm
    return math.cos(norm) * p + math.sin(norm) * direction


def log_sphere(p, q):
    p = np.asarray(p, dtype=float)
    q = np.asarray(q, dtype=float)
    cosine = float(np.clip(np.dot(p, q), -1.0, 1.0))
    theta = math.acos(cosine)
    if theta < 1e-12:
        return np.zeros(3)
    direction = (q - cosine * p) / math.sin(theta)
    return theta * direction


def sphere_geodesic(p, q, n=80):
    tangent = log_sphere(p, q)
    theta = np.linalg.norm(tangent)
    direction = tangent / theta
    ts = np.linspace(0, theta, n)
    return np.array([math.cos(t) * p + math.sin(t) * direction for t in ts])


p = np.array([0.0, 0.0, 1.0])
v_tangent = np.array([0.82, 0.38, 0.0])
q = exp_sphere(p, v_tangent)
recovered = log_sphere(p, q)
geodesic = sphere_geodesic(p, q, 100)

phi = np.linspace(0, 2 * np.pi, 80)
theta_grid = np.linspace(0, np.pi, 40)
Phi, Theta = np.meshgrid(phi, theta_grid)
X_s = np.sin(Theta) * np.cos(Phi)
Y_s = np.sin(Theta) * np.sin(Phi)
Z_s = np.cos(Theta)
plane_x = np.linspace(-1.0, 1.0, 2)
plane_y = np.linspace(-1.0, 1.0, 2)
PX, PY = np.meshgrid(plane_x, plane_y)
PZ = np.ones_like(PX)

fig = go.Figure()
fig.add_trace(go.Surface(x=X_s, y=Y_s, z=Z_s, opacity=0.34, showscale=False, colorscale="Blues", name="unit sphere"))
fig.add_trace(go.Surface(x=PX, y=PY, z=PZ, opacity=0.28, showscale=False, colorscale=[[0, "#31a354"], [1, "#31a354"]], name="tangent plane"))
fig.add_trace(go.Scatter3d(x=geodesic[:, 0], y=geodesic[:, 1], z=geodesic[:, 2], mode="lines", line={"color": "#fdae61", "width": 7}, name="geodesic"))
fig.add_trace(go.Scatter3d(x=[p[0], q[0]], y=[p[1], q[1]], z=[p[2], q[2]], mode="markers+text", marker={"size": 5, "color": ["#31a354", "#d95f0e"]}, text=["p", "exp_p(v)"], textposition="top center", name="points"))
fig.add_trace(go.Scatter3d(x=[p[0], p[0] + v_tangent[0]], y=[p[1], p[1] + v_tangent[1]], z=[p[2], p[2] + v_tangent[2]], mode="lines+markers", line={"color": "#31a354", "width": 6}, marker={"size": 3}, name="tangent vector v"))
fig.add_trace(go.Scatter3d(x=[p[0], p[0] + recovered[0]], y=[p[1], p[1] + recovered[1]], z=[p[2], p[2] + recovered[2]], mode="lines", line={"color": "#756bb1", "width": 4, "dash": "dash"}, name="log_p(exp_p(v))"))
fig.update_layout(
    title="Sphere geodesic, tangent plane, and exp/log maps",
    scene={"aspectmode": "data", "xaxis_title": "x", "yaxis_title": "y", "zaxis_title": "z"},
    margin={"l": 0, "r": 0, "t": 45, "b": 0},
    legend={"x": 0.02, "y": 0.98},
)
html_path = remember(save_plotly_html(fig, "chapter-04", "sphere-geodesics-exp-log-charts.html", include_plotlyjs=True))

checks["sphere_exp_log"] = {
    "tangent_dot_base_point": float(np.dot(v_tangent, p)),
    "q_norm": float(np.linalg.norm(q)),
    "geodesic_endpoint_error": float(np.linalg.norm(geodesic[-1] - q)),
    "log_exp_roundtrip_error": float(np.linalg.norm(recovered - v_tangent)),
    "geodesic_length_equals_tangent_norm": float(np.linalg.norm(v_tangent)),
}
assert abs(checks["sphere_exp_log"]["tangent_dot_base_point"]) < 1e-12
assert abs(checks["sphere_exp_log"]["q_norm"] - 1.0) < 1e-12
assert checks["sphere_exp_log"]["geodesic_endpoint_error"] < 1e-12
assert checks["sphere_exp_log"]["log_exp_roundtrip_error"] < 1e-12
display_artifact(html_path, width="100%", height=560)


## 6. Gauges: Coordinate Frames Are Choices, Not Geometry

A gauge at a point is a selected linear isomorphism from a model vector space such as `R^2` into the tangent space. On a surface, that means choosing a tangent basis. The same physical tangent vector can be represented by different coordinate pairs under different gauges. A gauge-aware operator must transform coordinates and features consistently, so the intrinsic vector or norm does not depend on the arbitrary local frame.


In [ ]:
alpha = math.radians(38)
R = np.array([[math.cos(alpha), -math.sin(alpha)], [math.sin(alpha), math.cos(alpha)]])
frame = np.array([[1.0, 0.0], [0.0, 1.0], [0.0, 0.0]])
rotated_frame = frame @ R
coords_in_rotated_gauge = np.array([0.92, 0.28])
coords_in_original_gauge = R @ coords_in_rotated_gauge
physical_from_original = frame @ coords_in_original_gauge
physical_from_rotated = rotated_frame @ coords_in_rotated_gauge

fig = plt.figure(figsize=(11, 4.6))
grid = fig.add_gridspec(1, 2, width_ratios=[1.1, 0.9])
ax = fig.add_subplot(grid[0, 0], projection="3d")
phi = np.linspace(0, 2 * np.pi, 64)
theta_grid = np.linspace(0, np.pi, 32)
Phi, Theta = np.meshgrid(phi, theta_grid)
ax.plot_surface(np.sin(Theta) * np.cos(Phi), np.sin(Theta) * np.sin(Phi), np.cos(Theta), color="#d8ecf6", alpha=0.45, linewidth=0)
base = np.array([0.0, 0.0, 1.0])
for vec, color, label in [(frame[:, 0], COLOR_A, "e1"), (frame[:, 1], COLOR_A, "e2"), (rotated_frame[:, 0], COLOR_B, "e1'"), (rotated_frame[:, 1], COLOR_B, "e2'")]:
    ax.quiver(*base, *(0.55 * vec), color=color, linewidth=2)
    ax.text(*(base + 0.62 * vec), label, color=color)
ax.quiver(*base, *(0.65 * physical_from_original), color=COLOR_C, linewidth=4)
ax.text(*(base + 0.72 * physical_from_original), "same tangent vector", color=COLOR_C)
ax.set_title("two gauges at the same tangent plane")
ax.set_box_aspect((1, 1, 1))
ax.view_init(elev=28, azim=-52)
ax.set_axis_off()

ax2 = fig.add_subplot(grid[0, 1])
ax2.set_title("coordinate rotation leaves intrinsic norm unchanged")
labels = ["rotated gauge", "original gauge"]
values = np.vstack([coords_in_rotated_gauge, coords_in_original_gauge])
xpos = np.arange(len(labels))
bar_width = 0.35
ax2.bar(xpos - bar_width / 2, values[:, 0], width=bar_width, color=COLOR_A, label="coordinate 1")
ax2.bar(xpos + bar_width / 2, values[:, 1], width=bar_width, color=COLOR_B, label="coordinate 2")
for i, row in enumerate(values):
    ax2.text(i, max(row) + 0.08, f"norm={np.linalg.norm(row):.3f}", ha="center")
ax2.set_xticks(xpos, labels)
ax2.set_ylim(-0.1, 1.25)
ax2.grid(axis="y", alpha=0.2)
ax2.legend(frameon=False)
fig.suptitle("Gauge transformations change coordinates, not the tangent vector", y=1.02)
fig.tight_layout()

path = remember(save_matplotlib(fig, "chapter-04", "gauge-frame-equivariance.png"))
plt.close(fig)

checks["gauge_equivariance"] = {
    "omega_prime_equals_omega_after_tau_error": float(np.linalg.norm(physical_from_original - physical_from_rotated)),
    "coordinate_norm_difference": float(abs(np.linalg.norm(coords_in_original_gauge) - np.linalg.norm(coords_in_rotated_gauge))),
    "rotation_determinant": float(np.linalg.det(R)),
}
assert checks["gauge_equivariance"]["omega_prime_equals_omega_after_tau_error"] < 1e-12
assert checks["gauge_equivariance"]["coordinate_norm_difference"] < 1e-12
assert abs(checks["gauge_equivariance"]["rotation_determinant"] - 1.0) < 1e-12
display_artifact(path, width=900)


## 7. Applied lab: A Tiny Manifold-Hypothesis Experiment

The manifold hypothesis says that meaningful high-dimensional observations often occupy a much lower-dimensional structured subset. The lab below builds a toy image family from two latent variables: horizontal pose and mouth curvature. Each image is a 16 by 16 vector in a 256-dimensional ambient pixel space. The images are synthetic, so the ground-truth latent dimension is known.

The local PCA check asks whether nearby valid samples have two dominant directions, while random ambient pixels do not. This does not prove the manifold hypothesis for real data. It shows why latent coordinates can be a useful modeling interface and why arbitrary points in ambient pixel space should not be expected to look valid.


In [ ]:
def synthetic_face(pose, smile, size=16):
    yy, xx = np.mgrid[-1:1:complex(size), -1:1:complex(size)]
    shift = 0.28 * pose
    head = np.exp(-(((xx - shift) / 0.72) ** 2 + (yy / 0.86) ** 2) * 1.6)
    left_eye = np.exp(-(((xx - shift + 0.24) / 0.08) ** 2 + ((yy + 0.2) / 0.08) ** 2) * 4.5)
    right_eye = np.exp(-(((xx - shift - 0.24) / 0.08) ** 2 + ((yy + 0.2) / 0.08) ** 2) * 4.5)
    mouth_curve = -0.35 + 0.28 * smile * ((xx - shift) ** 2 - 0.18)
    mouth = np.exp(-(((yy - mouth_curve) / 0.055) ** 2 + ((xx - shift) / 0.48) ** 4) * 2.2)
    img = 0.18 + 0.72 * head - 0.45 * left_eye - 0.45 * right_eye - 0.38 * mouth
    return np.clip(img, 0, 1)


poses = np.linspace(-1.0, 1.0, 13)
smiles = np.linspace(-1.0, 1.0, 13)
latents = np.array([(p0, s0) for p0 in poses for s0 in smiles])
images = np.array([synthetic_face(p0, s0).ravel() for p0, s0 in latents])

center_index = np.argmin(np.sum(latents**2, axis=1))
tree = cKDTree(latents)
neighbor_ids = tree.query(latents[center_index], k=31)[1]
local_valid = images[neighbor_ids] - images[neighbor_ids].mean(axis=0, keepdims=True)
random_pixels = rng.uniform(0, 1, size=local_valid.shape)
random_pixels = random_pixels - random_pixels.mean(axis=0, keepdims=True)
valid_s = svd(local_valid, compute_uv=False)
random_s = svd(random_pixels, compute_uv=False)
valid_var = valid_s**2 / np.sum(valid_s**2)
random_var = random_s**2 / np.sum(random_s**2)

pca_rows = [
    {"cloud": "valid_synthetic_images", "top_1_variance": float(valid_var[:1].sum()), "top_2_variance": float(valid_var[:2].sum()), "top_5_variance": float(valid_var[:5].sum())},
    {"cloud": "random_ambient_pixels", "top_1_variance": float(random_var[:1].sum()), "top_2_variance": float(random_var[:2].sum()), "top_5_variance": float(random_var[:5].sum())},
]
pca_path = remember(save_table_csv(pca_rows, "chapter-04", "manifold-hypothesis-local-dimension.csv"))

fig = plt.figure(figsize=(13, 6.2))
gs = fig.add_gridspec(3, 9, height_ratios=[0.15, 1, 1], hspace=0.25, wspace=0.04)
ax_title1 = fig.add_subplot(gs[0, 0:3]); ax_title1.axis("off"); ax_title1.text(0.5, 0.4, "latent traversal", ha="center", weight="bold")
ax_title2 = fig.add_subplot(gs[0, 3:6]); ax_title2.axis("off"); ax_title2.text(0.5, 0.4, "valid grid samples", ha="center", weight="bold")
ax_title3 = fig.add_subplot(gs[0, 6:9]); ax_title3.axis("off"); ax_title3.text(0.5, 0.4, "random ambient pixels", ha="center", weight="bold")

path_latents = np.column_stack([np.linspace(-0.9, 0.9, 6), 0.75 * np.sin(np.linspace(-1.2, 1.2, 6))])
for i, (p0, s0) in enumerate(path_latents):
    ax = fig.add_subplot(gs[1 + i // 3, i % 3])
    ax.imshow(synthetic_face(p0, s0), cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"({p0:.1f},{s0:.1f})", fontsize=8)
    ax.axis("off")

sample_pairs = [(-0.9, -0.9), (0.0, -0.9), (0.9, -0.9), (-0.9, 0.9), (0.0, 0.9), (0.9, 0.9)]
for i, (p0, s0) in enumerate(sample_pairs):
    ax = fig.add_subplot(gs[1 + i // 3, 3 + i % 3])
    ax.imshow(synthetic_face(p0, s0), cmap="gray", vmin=0, vmax=1)
    ax.axis("off")

for i in range(6):
    ax = fig.add_subplot(gs[1 + i // 3, 6 + i % 3])
    ax.imshow(rng.uniform(0, 1, size=(16, 16)), cmap="gray", vmin=0, vmax=1)
    ax.axis("off")
fig.suptitle("A two-parameter synthetic image manifold inside 256-dimensional pixel space", y=0.99)

path = remember(save_matplotlib(fig, "chapter-04", "manifold-hypothesis-latent-lab.png"))
plt.close(fig)

checks["manifold_hypothesis"] = {
    "ambient_dimension": int(images.shape[1]),
    "latent_dimension": 2,
    "valid_top2_variance": pca_rows[0]["top_2_variance"],
    "random_top2_variance": pca_rows[1]["top_2_variance"],
}
assert checks["manifold_hypothesis"]["ambient_dimension"] == 256
assert checks["manifold_hypothesis"]["valid_top2_variance"] > 0.8
assert checks["manifold_hypothesis"]["valid_top2_variance"] > checks["manifold_hypothesis"]["random_top2_variance"] * 2.0
display_artifact(path, width=950)
display_artifact(pca_path, width=900)


## Sanity Checks

The final checks are deliberately redundant. They assert that generated files exist and are nonempty, that static images are not blank, and that the core mathematical claims behind the artifacts passed numeric or symbolic tests.


In [ ]:
checks["artifact_count"] = len(artifact_paths)
checks["artifact_relative_paths"] = [str(path.relative_to(ARTIFACT_ROOT)).replace("\\", "/") for path in artifact_paths]
check_path = remember(save_json(checks, "chapter-04", "topology-manifold-invariants.json"))

records = assert_chapter_artifacts(artifact_paths)
for image_name in [
    "open-set-neighborhood-test.png",
    "continuity-preimage-open-sets.png",
    "homeomorphism-homotopy-tracker.png",
    "euler-characteristic-surface-invariants.png",
    "chart-transition-tangent-space.png",
    "gauge-frame-equivariance.png",
    "manifold-hypothesis-latent-lab.png",
]:
    assert_nonblank_image(ARTIFACT_ROOT / "figures" / image_name, min_std=1.0)

assert checks["open_sets"]["finite_topology_axioms"]["discrete"]
assert checks["continuity_preimages"]["square_preimage_components"] == [[-1.5, -0.5], [0.5, 1.5]]
assert checks["homeomorphism_homotopy"]["circle_max_h1_lifetime"] > checks["homeomorphism_homotopy"]["disk_max_h1_lifetime"]
assert {row["surface"]: row["euler"] for row in checks["euler_characteristic"]}["torus (one handle)"] == 0
assert checks["sphere_exp_log"]["log_exp_roundtrip_error"] < 1e-12
assert checks["gauge_equivariance"]["omega_prime_equals_omega_after_tau_error"] < 1e-12
assert checks["manifold_hypothesis"]["valid_top2_variance"] > 0.8

print(f"Sanity checks passed for {len(records)} generated artifacts.")
print("Invariant summary:", check_path.relative_to(BOOK_ROOT))
display_artifact(check_path, width=900)


## Takeaways

Topology is the language of continuity and equivalence before measurement enters. Differential geometry adds compatible local coordinates, tangent spaces, and smooth change-of-chart rules, while Riemannian geometry adds inner products that make lengths, angles, geodesics, exponential maps, and logarithmic maps computable. Gauge choices are local coordinate choices; a geometric model should transform with the gauge rather than accidentally depend on it. The manifold hypothesis imports this viewpoint into learning: high-dimensional observations may be better treated as samples from a structured low-dimensional domain, but this is a modeling assumption that needs diagnostics rather than a theorem for arbitrary data.
